# Lab 10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lweber89/geog510-test/blob/main/labs/geog510_lab_10_lweber89.ipynb)

## Overview

This lab introduces Google Earth Engine (GEE) for accessing and analyzing geospatial data in Python. You will explore diverse data types, including DEM, satellite imagery, and land cover datasets. You’ll gain experience creating cloud-free composites, visualizing temporal changes, and generating animations for time-series data.


## Objectives

By completing this lab, you will be able to:

1. Access and visualize Digital Elevation Model (DEM) data for a specific region.
2. Generate cloud-free composites from Sentinel-2 or Landsat imagery for a specified year.
3. Visualize National Agricultural Imagery Program (NAIP) data for U.S. counties.
4. Display watershed boundaries using Earth Engine styling.
5. Visualize land cover changes over time using split-panel maps.
6. Create a time-lapse animation for land cover changes over time in a region of your choice.


In [ ]:
%pip install -U "geemap[workshop]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 28.5 MB/s eta 0:00:00


In [2]:
import ee
import geemap
from google.colab import userdata

ee.Authenticate()

# 1. Grab your pre-saved secrets
# (Make sure these names match exactly what you typed in the Key icon menu)
ee_project = userdata.get('EARTHENGINE_PROJECT')
ee_token = userdata.get('EARTHENGINE_TOKEN')

# 2. Authenticate and Initialize in one move
try:
    # This attempts to use the token already in your Colab environment
    ee.Initialize(project=ee_project)
    print(f"✅ Connection Successful: Project '{ee_project}' is active.")
except Exception as e:
    # If the environment is fresh, we manually set the token
    import os
    os.environ['EARTHENGINE_TOKEN'] = ee_token
    ee.Initialize(project=ee_project)
    print(f"✅ Connection Successful: Initialized via Secret Token.")

✅ Connection Successful: Project 'project-0e9263ae-8f7a-423d-b2d' is active.


## Exercise 1: Visualizing DEM Data

Find a DEM dataset in the [Earth Engine Data Catalog](https://developers.google.com/earth-engine/datasets) and clip it to a specific area (e.g., your country, state, or city). Display it with an appropriate color palette. For example, the sample map below shows the DEM of the state of Colorado.


In [ ]:
#Pull dem from GEE

dem = ee.Image("USGS/SRTMGL1_003")

#Pull US States from GEE and create ROI for Montana
states = ee.FeatureCollection("TIGER/2018/States")
roi = states.filter(ee.Filter.eq('NAME', 'Montana'))

#Clip dem by roi

clipped_dem = dem.clip(roi)

#Initialize Map
m = geemap.Map()
m.center_object(roi, 6)

#Define visualization of dem

vis_params = {
    "min": 0,
    "max": 4000,
    "palette": ["006633", "E5FFCC", "662A00", "D8D8D8", "F5F5F5"],
}

#Create a state outline
state_outline = ee.Image().paint(roi, 0, 3)

#Add styled clipped_dem and state outline to map
m.add_layer(clipped_dem, vis_params, "SRTM DEM - Clipped")
m.add_layer(state_outline, {'palette': 'FF0000'}, 'State Outline Only')

#Add a text box with creator information
m.add_text(
    text="Created by lweber89",
    position='bottomright',
    padding='10px'
)

m

Map(center=[47.05213189349841, -109.63309161751589], controls=(WidgetControl(options=['position', 'transparent…

## Exercise 2: Cloud-Free Composite with Sentinel-2 or Landsat

Use Sentinel-2 or Landsat-9 data to create a cloud-free composite for a specific year in a region of your choice.

Use [Sentinel-2](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) or [Landsat-9 data](https://developers.google.com/earth-engine/datasets/catalog/landsat-9) data to create a cloud-free composite for a specific year in a region of your choice. Display the imagery on the map with a proper band combination.

In [ ]:
#Access image collection
collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")

#Pull US States from GEE and create ROI for Montana
states = ee.FeatureCollection("TIGER/2018/States")
roi = states.filter(ee.Filter.eq('NAME', 'Montana'))

#Collect images in 2024 that are cloud free, use the Montana ROI as the Geometry
images = (
    collection
    .filterBounds(roi.geometry())
    .filterDate("2024-01-01", "2024-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5))
)

#Check the result
count = images.size().getInfo()
print(f"Total images found for Montana in 2024: {count}")

#Create a state outline
state_outline = ee.Image().paint(roi, 0, 3)

#Initialize Map
m = geemap.Map()
m.center_object(roi, 6)

#Add styled state outline to map
m.add_layer(state_outline, {'palette': 'FF0000'}, 'State Outline Only')

m.addLayer(images.median().clip(roi), {'bands': ['B8', 'B4', 'B3'], 'max': 3000}, '2024 Median Composite')

#Add a text box with creator information
m.add_text(
    text="Created by lweber89",
    position='bottomright',
    padding='10px'
)

m

Total images found for Montana in 2024: 2162


Map(center=[47.05213189349841, -109.63309161751589], controls=(WidgetControl(options=['position', 'transparent…

## Exercise 3: Visualizing NAIP Imagery

Use [NAIP](https://developers.google.com/earth-engine/datasets/catalog/USDA_NAIP_DOQQ) imagery to create a cloud-free imagery for a U.S. county of your choice. Keep in mind that there might be some counties with the same name in different states, so make sure to select the correct county for the selected state.


In [ ]:
#Access image collection
collection = ee.ImageCollection("USDA/NAIP/DOQQ")

#Create ROI for St. Louis City, MO (a county-level equivelant)

counties = ee.FeatureCollection("TIGER/2018/Counties")
roi = counties.filter(ee.Filter.eq('GEOID', '29510'))

#Filter image collection for most recent STL NAIP imagery
stl_naip_images = collection.filterBounds(roi.geometry()).filterDate("2021-01-01", "2025-12-31")


#Create the composite using the median method

stl_recent_naip = stl_naip_images.median()

stl_final_comp = stl_recent_naip.clip(roi)

#Establish visualization parameters for NAIP
viz_params = {'bands': ['R', 'G', 'B'], 'min': 0, 'max': 255}

#Create a county outline
county_outline = ee.Image().paint(roi, 0, 3)

#Initialize Map
m = geemap.Map()
m.center_object(roi, 11)


#Add styled state outline to map
m.add_layer(county_outline, {'palette': 'FF0000'}, 'STL Outline')
m.add_layer(stl_final_comp, viz_params, 'St. Louis City NAIP 2023')


#Add a text box with creator information
m.add_text(
    text="Created by lweber89",
    position='bottomright',
    padding='10px'
)

m

Map(center=[38.63574898953896, -90.245188523643], controls=(WidgetControl(options=['position', 'transparent_bg…

## Exercise 4: Visualizing Watershed Boundaries

Visualize the [USGS Watershed Boundary Dataset](https://developers.google.com/earth-engine/datasets/catalog/USGS_WBD_2017_HUC04) with outline color only, no fill color.

In [ ]:
#Access collection
fc = ee.FeatureCollection("USGS/WBD/2017/HUC04")

#Initialize the map, center and zoom
m = geemap.Map(center=[40, -100], zoom=4.25)

#Create style
style = {'color': '0000ffff', 'width':2, "lineType": "solid", 'fillColor': '00000000'}

#Add styled feature collection to the map

m.add_layer(fc.style(**style), {}, "WBD 2017")

#Add a text box with creator information
m.add_text(
    text="Created by lweber89",
    position='bottomright',
    padding='10px'
)

m



Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…

## Exercise 5: Visualizing Land Cover Change

Use the [USGS National Land Cover Database](https://developers.google.com/earth-engine/datasets/catalog/USGS_NLCD_RELEASES_2019_REL_NLCD) and [US Census States](https://developers.google.com/earth-engine/datasets/catalog/TIGER_2018_States) to create a split-panel map for visualizing land cover change (2001-2019) for a US state of your choice. Make sure you add the NLCD legend to the map.

In [ ]:
#Pull NLCD from GEE

nlcd_2001 = ee.Image("USGS/NLCD_RELEASES/2019_REL/NLCD/2001").select("landcover")
nlcd_2021 = ee.Image("USGS/NLCD_RELEASES/2021_REL/NLCD/2021").select("landcover")


#Pull US States from GEE and create ROI
states = ee.FeatureCollection("TIGER/2018/States")
roi = states.filter(ee.Filter.eq('NAME', 'Florida'))

#Initialize Map
m = geemap.Map()
m.center_object(roi, 7)

#Populate left layer and right layer - clip by roi
left_layer = geemap.ee_tile_layer(nlcd_2001.clip(roi), {}, "NLCD 2001")
right_layer = geemap.ee_tile_layer(nlcd_2021.clip(roi), {}, "NLCD 2021")

m.split_map(left_layer, right_layer)

#Add a text box with creator information
m.add_text(
    text="Created by lweber89",
    position='topright',
    padding='10px'
)

#Add a text box with creator information
m.add_text(
    text="2001",
    position='bottomleft',
    padding='10px'
)

#Add a text box with creator information
m.add_text(
    text="2021",
    position='bottomright',
    padding='10px'
)

m.add_legend(
title="NLCD Land Cover Classification", builtin_legend="NLCD", height="455px")

m


Map(center=[28.460067698576307, -82.42296617464744], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## Exercise 6: Creating a Landsat Timelapse Animation

Generate a timelapse animation using Landsat data to show changes over time for a selected region.

In [ ]:
#Center point of roi - create BBox from pt
point = ee.Geometry.Point([-146.89, 61.13])
roi = point.buffer(20000).bounds()

#Configure and visualize timelapse

timelapse = geemap.landsat_timelapse(
roi,
out_gif="columbia_glacier.gif",
start_year=2000,
end_year=2025,
bands=["NIR", "Red", "Green"],
frames_per_second=5,
title="Columbia Glacier, AK",
font_color="blue",
)
geemap.show_image(timelapse)

Generating URL...
Please wait ...
The GIF image has been saved to: /content/columbia_glacier.gif


Output()

In [4]:
!pip install -qq ffmpeg-python

In [5]:

# Example: Dubai Urban Growth
point = ee.Geometry.Point([55.15, 25.12])
roi = point.buffer(20000).bounds()

timelapse = geemap.landsat_timelapse(
    roi,
    out_gif="dubai_growth.gif",
    start_year=2000,
    end_year=2025,
    bands=["SWIR1", "NIR", "Red"],  # Adjusted for urban/desert contrast
    frames_per_second=5,
    title="Dubai Expansion, UAE",
    font_color="white",
)
geemap.show_image(timelapse)

Generating URL...
Please wait ...
The GIF image has been saved to: /content/dubai_growth.gif


Output()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [7]:


# Center point of Wadi As-Sirhan agricultural region
# A 25km buffer captures a massive cluster of center-pivot fields
point = ee.Geometry.Point([38.35, 30.12])
roi = point.buffer(25000).bounds()

# Configure and visualize the desert agriculture timelapse
timelapse = geemap.landsat_timelapse(
    roi,
    out_gif="saudi_agriculture.gif",
    start_year=2000,
    end_year=2025,
    bands=["NIR", "Red", "Green"],  # False Color: Vegetation appears bright red
    frames_per_second=5,
    title="Desert Agriculture, Saudi Arabia",
    font_color="white",             # White font stands out well against dark water/crops
)

# Display the resulting GIF in Colab
geemap.show_image(timelapse)

Generating URL...
Please wait ...
The GIF image has been saved to: /content/saudi_agriculture.gif


Output()

In [8]:
# Center point of Longyangxia Dam Solar Park, China
# A 15km buffer will capture the massive, geometric layout of the panels
point = ee.Geometry.Point([100.62, 36.17])
roi = point.buffer(15000).bounds()

# Configure and visualize the solar farm expansion
timelapse = geemap.landsat_timelapse(
    roi,
    out_gif="china_solar_farm.gif",
    start_year=2000,
    end_year=2025,
    bands=["SWIR1", "NIR", "Red"],  # SWIR/NIR makes solar panels pop out as dark geometric tiles
    frames_per_second=5,
    title="Longyangxia Solar Park, China",
    font_color="white",
)

# Display the resulting GIF in Colab
geemap.show_image(timelapse)

Generating URL...
Please wait ...
The GIF image has been saved to: /content/china_solar_farm.gif


Output()